# RL_team (`dev` branch) — 통합 테스팅 노트북

이 노트북은 [`exGDGD/RL_team`](https://github.com/exGDGD/RL_team/tree/dev) 의 `dev` 브랜치를 대상으로
테스트를 한 번에 수행합니다. **Google Colab** 과 **로컬(Jupyter)** 양쪽에서 동작합니다.

프로젝트 개요: SimPy(비동기 이벤트 시뮬레이션) + PettingZoo(멀티에이전트 API) + Gymnasium(관측/행동 공간)
기반의 **이종(Performance/Efficiency) CPU 스케줄러** 멀티에이전트 강화학습 프레임워크.

> **최근 변경 (6/3 패치 — preemption):** busy 코어가 실행 중인 태스크를 멈추고 다른 태스크로
> 갈아탈 수 있는 **선점(preemption)** 을 추가했습니다(`enable_preemption`, 기본 on).
> wakeup 게이트(P1 우선순위 / P3 기아 + min_run), 컨텍스트 스위치 비용, 부분 burst 처리,
> 그리고 **NO-OP 을 학습 가능한 행동으로 기록**합니다. `self` 관측은 현재 실행 task 정보를
> 포함해 **8-dim** 으로 확장됐습니다.

## 노트북 구성
1. 저장소 클론 (`dev` 브랜치) & 작업 디렉터리 설정
2. 의존성 설치 + 임포트 sanity 체크
3. **전체 pytest 스위트** 실행
4. 모듈별 테스트 (env / baselines / rl) 개별 실행
5. 베이스라인 정책 평가 (`evaluate_baselines`)
6. 환경/관측/롤아웃 스모크 (PyTorch 불필요)
7. **Preemption & NO-OP 스모크** (PyTorch 불필요)
8. 신경망 · 트레이너 · 학습 루프 스모크 (PyTorch 필요)
9. 결과 요약

## 1. 저장소 클론 & 작업 디렉터리 설정

`dev` 브랜치를 클론합니다. 이미 클론되어 있으면 `git fetch` 후 최신으로 갱신합니다.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/exGDGD/RL_team.git"
# 클론/체크아웃할 브랜치 — Colab 폼에서 바로 수정하거나 환경변수 RL_TEAM_BRANCH 로 덮어쓰기 가능
BRANCH = "dev"  #@param {type:"string"}
BRANCH = os.environ.get("RL_TEAM_BRANCH", BRANCH)

# Colab 이면 /content, 아니면 현재 작업 폴더 기준
IN_COLAB = "google.colab" in sys.modules
BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = BASE_DIR / "RL_team"

def run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
else:
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH])

os.chdir(REPO_DIR)
# 'src' 패키지 임포트가 가능하도록 저장소 루트를 sys.path 에 추가
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("\n작업 디렉터리:", Path.cwd())
run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"])

## 2. 의존성 설치 & 임포트 sanity 체크

`requirements.txt` 의 패키지(gymnasium / pettingzoo / simpy / numpy / pytest)를 설치합니다.
RL 학습부에는 `torch` 가 필요합니다. Colab 에는 기본 설치되어 있고, 로컬에서 없으면 학습 셀만
건너뜁니다(나머지는 그대로 동작).

In [ ]:
# 핵심 의존성 설치
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# torch 가용성 확인 (학습/신경망 셀에서 사용)
try:
    import torch
    HAS_TORCH = True
    print("torch", torch.__version__)
except ModuleNotFoundError:
    HAS_TORCH = False
    print("[warn] torch 미설치 — 신경망/트레이너/학습 셀은 건너뜁니다.")
    print("       로컬에서 학습까지 테스트하려면:  pip install torch")

torch 2.11.0+cu128


In [ ]:
# 패키지 임포트 sanity 체크
import simpy, pettingzoo, gymnasium, numpy
print("simpy", simpy.__version__)
print("pettingzoo", pettingzoo.__version__)
print("gymnasium", gymnasium.__version__)
print("numpy", numpy.__version__)

# 프로젝트 모듈 임포트 확인
from src.env import CoreType, SchedulerEnv, WorkloadScenario
from src.rl import build_agent_batch, collect_episode, AgentBatch
from src.baselines import RandomPolicy, RoundRobinPolicy, SJFLikePolicy, EASLikePolicy, run_episode
print("\n[ok] 모든 핵심 모듈 임포트 성공")

simpy 4.1.1
pettingzoo 1.24.3
gymnasium 1.0.0
numpy 2.0.2

[ok] 모든 핵심 모듈 임포트 성공


## 3. 전체 pytest 스위트 실행

`tests/` 의 모든 테스트를 실행합니다 (env / metrics / baselines / rl 버퍼·네트워크·관측·롤아웃·트레이너).

> 참고: `test_rl_networks.py` 와 `test_rl_trainer.py` 는 PyTorch 가 필요합니다. torch 가 없으면
> 해당 테스트는 import 단계에서 실패/스킵될 수 있으므로, torch 유무에 따라 대상 파일을 조정합니다.

In [ ]:
test_args = [sys.executable, "-m", "pytest", "-q", "--maxfail=0"]

if HAS_TORCH:
    # 전체 스위트
    test_args.append("tests/")
else:
    # torch 비의존 테스트만
    test_args += [
        "tests/test_scheduler_env.py",
        "tests/test_metrics.py",
        "tests/test_baselines.py",
        "tests/test_rl_obs.py",
        "tests/test_rl_buffer.py",
        "tests/test_rl_rollout.py",
    ]

result = subprocess.run(test_args)
print("\npytest exit code:", result.returncode, "(0 이면 전부 통과)")


pytest exit code: 1 (0 이면 전부 통과)


## 4. 모듈별 테스트 개별 실행 (디버깅용)

전체 실행에서 실패가 있을 때, 어느 모듈인지 빠르게 좁히기 위한 셀입니다. `-v` 로 상세 출력합니다.

In [ ]:
module_tests = {
    "env":       ["tests/test_scheduler_env.py", "tests/test_metrics.py"],
    "baselines": ["tests/test_baselines.py"],
    "rl (no torch)": ["tests/test_rl_obs.py", "tests/test_rl_buffer.py", "tests/test_rl_rollout.py"],
    "rl (torch)":    ["tests/test_rl_networks.py", "tests/test_rl_trainer.py"],
}

for label, files in module_tests.items():
    if label == "rl (torch)" and not HAS_TORCH:
        print(f"\n===== {label} : torch 없음 -> 스킵 =====")
        continue
    print(f"\n===== {label} =====")
    subprocess.run([sys.executable, "-m", "pytest", "-v", *files])


===== env =====

===== baselines =====

===== rl (no torch) =====

===== rl (torch) =====


## 5. 베이스라인 정책 평가

`Random / RoundRobin / SJF-like / EAS-like` 4개 정책을 4개 워크로드 시나리오
(BALANCED / UI_HEAVY / BG_HEAVY / BURST_STRESS)에서 평가하여 보상·처리량·에너지·응답시간 등을
표로 출력합니다. **PyTorch 불필요.**

In [ ]:
subprocess.run([sys.executable, "-m", "src.evaluate_baselines"])

CompletedProcess(args=['/usr/bin/python3', '-m', 'src.evaluate_baselines'], returncode=0)

## 6. 환경 / 관측 / 롤아웃 스모크 (PyTorch 불필요)

환경을 직접 만들고, 관측을 에이전트 배치로 변환한 뒤, 간단한 규칙 기반 정책으로 한 에피소드를
수집해 봅니다.

In [ ]:
env = SchedulerEnv(
    core_config={CoreType.P: 1, CoreType.E: 1},
    workload_scenario=WorkloadScenario.BALANCED,
    arrival_rate=0.5,
    episode_time=30.0,
    max_tasks=4,
    seed=3,
)
observations, info = env.reset()
batch = build_agent_batch(observations, agent_order=env.agents)

print("agents:        ", batch.agent_ids)
print("self_features: ", batch.self_features.shape)
print("ready_queue:   ", batch.ready_queue.shape)
print("other_cores:   ", batch.other_cores.shape)
print("action_mask:   ", batch.action_mask.shape)
print("decision agents:", batch.decision_agent_ids())

agents:         ('p_0', 'e_0')
self_features:  (2, 5)
ready_queue:    (2, 8, 6)
other_cores:    (2, 1, 3)
action_mask:    (2, 9)
decision agents: ('p_0', 'e_0')


In [ ]:
class FirstValidPolicy:
    """각 코어에서 첫 번째 유효한(인덱스>0) 행동을 선택하는 규칙 기반 정책."""
    def act(self, batch: AgentBatch):
        actions, log_probs = {}, {}
        for row, agent_id in enumerate(batch.agent_ids):
            valid = [idx for idx, ok in enumerate(batch.action_mask[row]) if idx > 0 and ok]
            actions[agent_id] = valid[0] if bool(batch.decision_mask[row]) and valid else 0
            log_probs[agent_id] = 0.0
        return actions, log_probs

env = SchedulerEnv(
    core_config={CoreType.P: 1, CoreType.E: 1},
    workload_scenario=WorkloadScenario.BALANCED,
    arrival_rate=0.5,
    episode_time=30.0,
    max_tasks=4,
    seed=3,
)
buffer = collect_episode(env, FirstValidPolicy(), seed=3)

print("transitions:                ", len(buffer))
print("completed tasks:            ", len(env.completed_tasks))
print("first transition elapsed_time:", buffer.transitions[0].elapsed_time)

m = env.metrics()
print("\n[metrics] completed/total:", m.completed_tasks, "/", m.total_tasks)
print("[metrics] throughput:     ", round(m.throughput, 4))

transitions:                 8
completed tasks:             4
first transition elapsed_time: 6.122869919011805

[metrics] completed/total: 4 / 4
[metrics] throughput:      0.1635


## 7. Preemption & NO-OP 스모크 (PyTorch 불필요)

선점이 켜진 환경(`enable_preemption=True`)에서 busy 코어가 실제로 실행 중 태스크를 멈추고
다른 태스크로 갈아타는지(부분 burst + 컨텍스트 스위치 비용), 그리고 **NO-OP 이 학습 트랜지션으로
기록**되는지(item 2) 확인합니다. 선점 ON/OFF 를 같은 시드로 비교합니다.

In [ ]:
# --- (a) 선점 동작 확인: busy 코어가 실행 중 태스크를 멈추고 갈아탐 ---
preempt_env = SchedulerEnv(
    core_config={CoreType.E: 1},          # 느린 코어 1개 -> 긴 실행, 선점 유도가 쉬움
    workload_scenario=WorkloadScenario.BURST_STRESS,
    arrival_rate=3.0,
    episode_time=60.0,
    max_tasks=40,
    seed=5,
    enable_preemption=True,
    preempt_min_run=0.0,                  # 최소 실행 가드 끔(데모용)
)
buf_on = collect_episode(preempt_env, FirstValidPolicy(), seed=5)
total_preemptions = sum(task.preemptions for task in preempt_env.tasks.values())
print("[preemption ON]")
print("  누적 선점 횟수:   ", total_preemptions)
print("  transitions:      ", len(buf_on))
print("  completed tasks:  ", len(preempt_env.completed_tasks))

# 같은 시드, 선점 OFF (ablation 비교)
off_env = SchedulerEnv(
    core_config={CoreType.E: 1},
    workload_scenario=WorkloadScenario.BURST_STRESS,
    arrival_rate=3.0, episode_time=60.0, max_tasks=40, seed=5,
    enable_preemption=False,
)
buf_off = collect_episode(off_env, FirstValidPolicy(), seed=5)
print("\n[preemption OFF]")
print("  누적 선점 횟수:   ", sum(t.preemptions for t in off_env.tasks.values()))
print("  transitions:      ", len(buf_off))

# --- (b) NO-OP 기록 확인: 아무것도 안 하는 정책도 트랜지션이 남음 (item 2) ---
class NoOpPolicy:
    def act(self, batch):
        return ({a: 0 for a in batch.agent_ids}, {a: 0.0 for a in batch.agent_ids})

noop_env = SchedulerEnv(
    core_config={CoreType.P: 1},
    workload_scenario=WorkloadScenario.BALANCED,
    arrival_rate=0.5, episode_time=30.0, max_tasks=4, seed=3,
)
noop_buf = collect_episode(noop_env, NoOpPolicy(), seed=3, max_env_steps=5)
print("\n[NO-OP 정책]")
print("  기록된 NO-OP 트랜지션:", len(noop_buf),
      "| 모두 action==0:", all(t.action == 0 for t in noop_buf.transitions))

assert total_preemptions > 0, "선점이 한 번도 발생하지 않았습니다 (시나리오/시드 조정 필요)"
assert len(noop_buf) > 0, "NO-OP 이 트랜지션으로 기록되지 않았습니다 (item 2)"
print("\n[ok] 선점·NO-OP 스모크 통과")

## 8. 신경망 · 트레이너 · 학습 루프 스모크 (PyTorch 필요)

아래 셀들은 torch 가 있을 때만 실행됩니다. `TypeSharedActor` / `AgentCentricCritic` 순전파,
`ACACTrainer.update` 한 스텝, 그리고 `src.train_acac` 엔드투엔드 학습을 짧게 돌려봅니다.

> 학습은 이제 **선점이 기본 on** 입니다. 비선점(ablation)으로 돌리려면 `--disable-preemption` 을
> 붙이세요. `allow_noop` 도 기본 True 로 바뀌어 NO-OP/keep 이 학습 행동에 포함됩니다.

In [ ]:
if not HAS_TORCH:
    print("torch 없음 -> 7장 스킵")
else:
    import torch
    from src.rl.networks import AgentCentricCritic, TypeSharedActor

    def as_tensor(array, dtype=torch.float32):
        return torch.as_tensor(array, dtype=dtype)

    actor = TypeSharedActor(hidden_dim=64)
    critic = AgentCentricCritic(hidden_dim=64, num_heads=4)

    with torch.no_grad():
        logits = actor(
            self_features=as_tensor(batch.self_features),
            ready_queue=as_tensor(batch.ready_queue),
            ready_mask=as_tensor(batch.ready_mask),
            other_cores=as_tensor(batch.other_cores),
            other_core_mask=as_tensor(batch.other_core_mask),
            system=as_tensor(batch.system),
            action_mask=as_tensor(batch.action_mask, dtype=torch.bool),
        )
        values = critic(
            self_features=as_tensor(batch.self_features),
            ready_queue=as_tensor(batch.ready_queue),
            ready_mask=as_tensor(batch.ready_mask),
            other_cores=as_tensor(batch.other_cores),
            other_core_mask=as_tensor(batch.other_core_mask),
            system=as_tensor(batch.system),
        )

    print("logits:", logits.shape)
    print("values:", values.shape)
    print("sample logits row:", [round(x, 3) for x in logits[0].tolist()])

logits: torch.Size([2, 9])
values: torch.Size([2])
sample logits row: [-0.052, -0.918, -1000000000.0, -1000000000.0, -1000000000.0, -1000000000.0, -1000000000.0, -1000000000.0, -1000000000.0]


In [ ]:
if not HAS_TORCH:
    print("torch 없음 -> 트레이너 스킵")
else:
    from src.rl.trainer import ACACConfig, ACACTrainer, TorchACACPolicy

    env = SchedulerEnv(
        core_config={CoreType.P: 1, CoreType.E: 1},
        workload_scenario=WorkloadScenario.BALANCED,
        arrival_rate=0.5,
        episode_time=30.0,
        max_tasks=4,
        seed=3,
    )
    rollout = collect_episode(env, FirstValidPolicy(), seed=3)
    policy = TorchACACPolicy(ACACConfig(hidden_dim=64, critic_heads=4))
    trainer = ACACTrainer(policy)
    stats = trainer.update(rollout)
    print("update stats:")
    print("  loss:      ", round(stats.loss, 4))
    print("  value_loss:", round(stats.value_loss, 4))
    print("  entropy:   ", round(stats.entropy, 4))

update stats:
  loss:       0.1764
  value_loss: 0.0027
  entropy:    0.6912


In [ ]:
if not HAS_TORCH:
    print("torch 없음 -> 엔드투엔드 학습 스킵")
else:
    # 짧은 엔드투엔드 학습 (sanity). 시간이 걸리면 episodes 를 줄이세요.
    subprocess.run([
        sys.executable, "-m", "src.train_acac",
        "--episodes", "3",
        "--eval-every", "1",
        "--episode-time", "40",
        "--max-tasks", "8",
        "--hidden-dim", "32",
    ])

## 9. 결과 요약

- 3장 pytest exit code 가 `0` 이면 전체 테스트 통과.
- 5장 베이스라인 표가 출력되면 환경·정책·메트릭 파이프라인 정상.
- 6장 트랜지션 수집이 정상이면 관측/롤아웃 정상.
- **7장 선점·NO-OP 스모크가 `[ok]` 로 끝나면** 선점(부분 burst·CS 비용)과 NO-OP 기록 정상.
- 8장(torch) 셀들이 에러 없이 통과하면 액터/크리틱·트레이너·학습 루프 정상.

필요 시 `arrival_rate`, `episode_time`, `max_tasks`, `core_config`, `WorkloadScenario`,
`enable_preemption`, `--episodes`, `--disable-preemption` 등을 바꿔가며 추가 테스트를 진행하세요.

In [ ]:
# 300
#   --advantage-norm per_scenario: 시나리오별 advantage 표준화(burst 독점 방지). global 로 A/B.
#   --lr-anneal-final-frac 0.1: LR을 후반 10%까지 선형 감쇠 → KL/clip 우하향, 정책이 가라앉음(A1).
#   --entropy-coef-final 0.005: entropy 바닥(B1).
#   eval/test 표본 증량: --eval-episodes 40(시나리오당 10) --eval-every 20, --test-episodes 80(시나리오당 20).
#     시나리오당 2~3개는 너무 적어 best 선택/score가 노이즈에 휘둘렸음(특히 burst 2-seed).
#   best.pt = 시나리오별 (rl - best_baseline)/|best_baseline| 의 균등 평균(0=best heuristic)으로 선택.
!python -m src.train_acac \
    --reward-mode latency_flow \
    --lambda-flow 1.0 --response-weight 1.5 --lambda-energy 0 \
    --lambda-context-switch 3 \
    --episodes 300 --eval-every 20 --eval-episodes 40 --test-episodes 80 --seed 3 \
    --arrival-rate 1.0 --episode-time 40 --max-tasks 32 \
    --hidden-dim 128 --rollout-episodes 16 \
    --clip-ratio 0.2 --entropy-coef 0.01 --entropy-coef-final 0.005 --update-epochs 4 \
    --lr-anneal-final-frac 0.1 \
    --num-minibatches 4 --replay-capacity 0 \
    --advantage-norm per_scenario \
    --rollout-workers 2 --device cpu \
    --output-dir outputs/acac_flow

In [ ]:
# 학습 곡선 (확대 가능 · 인터랙티브 Plotly): 줌·팬·더블클릭 autoscale·호버·범례토글
#   패널: reward(대 baseline)+balanced_score / loss / entropy+coef / KL+clip / grad /
#         decision-mix / 시나리오별 reward(det·sampled) / 시나리오별 turnaround·throughput
# git fetch 로 코드 갱신했다면 아래 reload 로 새 src.plot_metrics 반영(런타임 재시작 없이).
import importlib
import src.plot_metrics as pm
importlib.reload(pm)

rows = pm.load_metrics("outputs/acac_flow/metrics.jsonl")

# 인터랙티브(확대 가능) — Colab inline 표시 + HTML 저장(브라우저에서 열어 더 크게 확대).
pm.plot_training_metrics_interactive(
    rows,
    title="acac_flow (300 iter)",
    reward_floor=None,        # 자동 하한(권장). 그래프에서 더블클릭하면 언제든 autoscale.
    save_html="outputs/acac_flow/metrics.html",
    show=True,
)

# 정적 PNG도 리포트용으로 저장(확대는 위 인터랙티브 사용).
pm.plot_training_metrics(rows, save_path="outputs/acac_flow/metrics.png",
                         show=False, title="acac_flow (300 iter)")
print("iterations:", len(rows),
      "| html=outputs/acac_flow/metrics.html  png=outputs/acac_flow/metrics.png")

In [ ]:
# 시나리오별 결과 표 (확대 가능한 HTML 표): RL(det/sampled) vs baselines
#   reward 외에 turnaround / response / throughput 도 함께 표로 출력.
from src.plot_metrics import (
    load_metrics, latest_eval_summary, scenario_metric_table,
    summarize_scenario_significance,
)

rows = load_metrics("outputs/acac_flow/metrics.jsonl")
cols = ["rl", "rl_sampled", "random", "mlfq", "sjf_like", "eas_like"]
# (key, 라벨) — reward·throughput 은 높을수록↑, turnaround·response 는 낮을수록↑.
METRICS = [
    ("reward",     "reward  (↑ 높을수록 좋음)"),
    ("turnaround", "mean turnaround  (↓ 낮을수록 좋음)"),
    ("response",   "mean response  (↓ 낮을수록 좋음)"),
    ("throughput", "throughput  (↑ 높을수록 좋음)"),
]

def show_metric_tables(summary, *, source=""):
    """eval-summary(학습로그 evaluation 또는 eval_checkpoint json)에서 지표별 표 출력."""
    if not summary or not summary.get("by_scenario"):
        print(f"[{source}] 시나리오별 데이터 없음 (단일 시나리오 학습이거나 옛 로그).")
        return
    try:
        import pandas as pd
        from IPython.display import display
        for key, label in METRICS:
            tbl = scenario_metric_table(summary, key)
            if not any(any(v is not None for v in r.values()) for r in tbl.values()):
                continue  # 이 지표가 로그에 없음(옛 로그) → 건너뜀
            df = pd.DataFrame(tbl).T[cols].apply(pd.to_numeric, errors="coerce")
            print(f"[{source}] {label}   (sjf_like* = clairvoyant oracle, 공정 비교 아님)")
            display(df.round(2))
    except Exception as e:  # pandas/IPython 없으면 텍스트로
        print("표 표시 실패, 텍스트로 출력:", e)
        for key, label in METRICS:
            tbl = scenario_metric_table(summary, key)
            print(f"\n[{source}] {label}")
            print("scenario".ljust(14) + "".join(c.rjust(11) for c in cols))
            for name, r in tbl.items():
                print(name.ljust(14) + "".join(
                    ("-" if r[c] is None else f"{r[c]:.1f}").rjust(11) for c in cols))

# 최신 eval 기준 표 (held-out test 가 있으면 아래 eval_checkpoint 셀의 고표본 표가 더 정확)
show_metric_tables(latest_eval_summary(rows), source="latest eval")

# RL vs '현실' best baseline(random/mlfq/eas) + SJF oracle 천장 (유의성; reward 기준)
#   WIN/LOSE = 표본이 차이를 분별(|Δ|>95%CI). tie = 미분별. SJF는 oracle라 목표 아님(gap만 참고).
test_summary = next((r["test"] for r in rows if r.get("split") == "test"), None)
target, label = (test_summary, "held-out test") if test_summary else (latest_eval_summary(rows), "latest eval")
sig = summarize_scenario_significance(target) if target else {}
if not sig:
    print("\n[분석] std/n 또는 현실 baseline 정보가 없는 로그입니다 (증량 설정으로 재학습하면 표시).")
else:
    print(f"\n[분석] {label} — RL vs 현실 best baseline (SJF=oracle, 목표 아님):")
    for sc in sorted(sig):
        s = sig[sc]
        verd = "?" if s["significant"] is None else ("WIN" if s["significant"] and s["delta"] > 0
               else "LOSE" if s["significant"] else "tie")
        orc = "" if s["oracle"] is None else f" | SJF* {s['oracle']:+.1f} gap {s['oracle_gap']:+.1f}"
        print(f"  {sc:<13} n={s['n']:<4} rl {s['rl']:+8.1f}±{s['rl_se']:4.1f} vs "
              f"{s['best_baseline']:<8} {s['best']:+8.1f} | Δ {s['delta']:+7.1f} ±{s['half_ci']:5.1f} {verd}{orc}")

In [ ]:
# best.pt 고표본 재평가 (재학습 X) — 최종 숫자 신뢰도 ↑
#   --episodes N = 시나리오당 N개(라운드로빈 총합 아님). 노이즈 큰 burst는 더 많이.
#   seed 30000 은 train/eval/test(≈4k/10k/20k)와 분리된 held-out.
#   sig=NO 인 시나리오만 episode를 더 늘리면 됨(예: burst만 키우기). 결과는 json 으로도 저장.
!python -m src.eval_checkpoint outputs/acac_flow/best.pt \
    --scenario-episodes balanced=80,bg_heavy=60,burst_stress=200,ui_heavy=60 \
    --seed 30000 --device cpu \
    --out outputs/acac_flow/eval_checkpoint.json

In [ ]:
# 최종 고표본 표 (eval_checkpoint.json) — reward/turnaround/response/throughput, 확대 가능한 HTML 표.
#   위 'acac-flow-scenario-table' 셀을 먼저 실행해야 함(show_metric_tables/METRICS/cols 정의).
import json

ckpt_json = "outputs/acac_flow/eval_checkpoint.json"
try:
    final_summary = json.load(open(ckpt_json, encoding="utf-8"))
    show_metric_tables(final_summary, source="held-out (eval_checkpoint)")
except FileNotFoundError:
    print(f"{ckpt_json} 없음 — 위 eval_checkpoint 셀을 먼저 실행하세요.")
except NameError:
    print("show_metric_tables 미정의 — 위 시나리오 표 셀(acac-flow-scenario-table)을 먼저 실행하세요.")

In [ ]:
# (선택 · 오래 걸림) 멀티시드 = 재현성/결론용  ── warm 은 제외(아래 참고)
#   scratch × 여러 시드를 각각 전체 학습 → 각 best.pt 를 held-out(시드 30000)에서 평가
#   → 시드 간 mean±std 집계. "burst=SJF급 접근", "balanced/bg 현실 best와 동률"이 1회 운인지 재현인지 판정.
#   분석은 oracle-aware: SJF(clairvoyant)는 천장(목표 아님), 비교 대상은 현실 baseline(random/mlfq/eas).
#   ⚠️ warm 제외 이유: cold-critic + 날카로운 BC-actor 조합이 PPO 첫 업데이트에서 불안정 →
#      멀티시드 3/3 시드가 all-noop 으로 붕괴(reward ~ -32000, 현실 baseline의 40배). 결과 아님(아티팩트).
#   best.pt 가 이미 있으면 재사용(재학습은 --force). 빠른 점검: --episodes 40 --seeds 0,1 로 축소.
!python -m src.multiseed --seeds 0,1,2 --configs scratch \
    --episodes 300 \
    --eval-scenario-episodes balanced=80,bg_heavy=80,burst_stress=200,ui_heavy=80 \
    --eval-seed 30000 --workers 2 --device cpu \
    --out outputs/multiseed

In [ ]:
#!python -m src.train_acac --episodes 100 --eval-every 5 --seed 3  --arrival-rate 1.0 --episode-time 40 --max-tasks 32 --hidden-dim 64 --rollout-episodes 16 --resume outputs/acac_p2e2/best.pt

In [ ]:
!ls -lh outputs/acac_flow
!tail -n 3 outputs/acac_flow/metrics.jsonl

In [ ]:
!python -m src.train_sjf_imitation --device cpu --output outputs/sjf_imitation/actors.pt

In [ ]:
!python -m src.train_acac \
  --device cpu \
  --episodes 30 \
  --eval-every 1 \
  --episode-time 40 \
  --max-tasks 32 \
  --pretrained-actors outputs/sjf_imitation/actors.pt \
  --output-dir outputs/acac_sjf_warm_start

In [ ]:
# 300 (warm start) — ⚠️ Colab 실행 제외 (결과 아님, 아티팩트)
#   cold-critic + 날카로운 BC-actor 조합이 PPO 첫 업데이트에서 불안정 → 멀티시드 3/3 시드가
#   all-noop 으로 붕괴(reward ~ -32000, 현실 baseline의 40배). "warm이 진다"는 결론 금지.
#   참고용으로 명령만 남겨둠. 다시 시도하려면 critic-warmup(첫 N iter actor freeze) 도입 후 재실행할 것.
# !python -m src.train_acac \
#     --reward-mode latency_flow \
#     --lambda-flow 1.0 --response-weight 1.5 --lambda-energy 0 \
#     --lambda-context-switch 3 \
#     --episodes 300 --eval-every 20 --eval-episodes 40 --test-episodes 80 --seed 3 \
#     --arrival-rate 1.0 --episode-time 40 --max-tasks 32 \
#     --hidden-dim 128 --rollout-episodes 16 \
#     --pretrained-actors outputs/sjf_imitation/actors.pt \
#     --clip-ratio 0.2 --entropy-coef 0.01 --entropy-coef-final 0.005 --update-epochs 4 \
#     --lr-anneal-final-frac 0.1 \
#     --num-minibatches 4 --replay-capacity 0 \
#     --advantage-norm per_scenario \
#     --rollout-workers 2 --device cpu \
#     --output-dir outputs/acac_flow_warm